# 10 · ASR baseline (T1_Speech)

Whisper large-v3-turbo on the **dev** audio → `asr/dev_en_whisper-large-v3-turbo.jsonl`
(one row per clip: `id, transcript, wer`). WER is computed against the text `question` from `qa_*` with the same `id`.

Backend is picked by `default_backend()`: `mlx` on Apple Silicon, `hf` on a CUDA GPU (Kaggle),
`ct2` (faster-whisper, int8 CPU) otherwise, e.g. on an Intel Mac. All three decode greedily.
Re-running skips clips that are already transcribed.

In [4]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from common import *
from speech import *

print("ASR backend:", default_backend())

ImportError: cannot import name 'LANG' from 'common' (/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/common/__init__.py)

## 1. Data check
Expected: 1000 rows, 0 duplicate ids, 0 missing audio/images.

In [ ]:
rows = load_split("sqa", "dev")
print(len(rows), check_ids(rows))
print(check_media(rows))
print(rows[0])

## 2. Smoke test on 5 clips
Compare by eye with the text question before the full run.

In [ ]:
sample = rows[:5]
refs = wer_refs("dev")
hyps = transcribe([r["audio_path"] for r in sample])
for r, h in zip(sample, hyps):
    print(f"WER {wer_one(h, refs[r['id']]):.2f}")
    print("  ref:", refs[r["id"]])
    print("  hyp:", h)

## 3. Full dev run

In [ ]:
ASR_MODEL = "whisper-large-v3-turbo"
path = run_asr("dev", model=ASR_MODEL)
path

In [ ]:
summary = asr_summary(path, "dev")
summary

## 4. Where does ASR fail?

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)

asr = pd.DataFrame(load_jsonl(path))
asr["question"] = asr["id"].map(refs)
asr["wer_bin"] = asr["wer"].map(wer_bin)
print(asr["wer_bin"].value_counts().reindex([b[2] for b in WER_BINS]).fillna(0).astype(int))
asr.sort_values("wer", ascending=False)[["wer", "question", "transcript"]].head(15)

Capitalized words (names, places, dishes) lost by ASR: the cultural part of the question.

In [ ]:
asr["missed"] = [missed_names(q, t) for q, t in zip(asr["question"], asr["transcript"])]
has = asr["missed"].map(len) > 0
print(f"questions with a lost name: {has.mean():.1%}")
pd.Series([w for ws in asr["missed"] for w in ws]).value_counts().head(25)